# Trích xuất đồ thị mã nguồn cho tập dữ liệu DiverseVul

Quy trình trích xuất hai biểu diễn đồ thị cho 9.077 hàm:
- Graph A: đồ thị cây cú pháp trừu tượng AST
- Graph B: đồ thị ngữ nghĩa kết hợp AST, luồng điều khiển CFG và phụ thuộc dữ liệu DATA_DEP

## 1. Môi trường

Internet để On. Accelerator chọn CPU. Dataset input là diversevul-raw.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "feature/3_3_graph-extract"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
REPO_DIR = Path("/kaggle/working") / "software-bug-detection-using-graphs"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(["git", *args], cwd=cwd, check=True)

if not REPO_DIR.exists():
    git(["clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=str(REPO_DIR))
    git(["checkout", GIT_BRANCH], cwd=str(REPO_DIR))
    git(["pull", "origin", GIT_BRANCH], cwd=str(REPO_DIR))

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
!pip install -q pyyaml

## 2. Cài đặt OpenJDK 21 và Joern

Cài đặt máy ảo Java và công cụ phân tích tĩnh Joern. Phiên bản Joern được ghim bằng `JOERN_VERSION` vì installer không có tham số `--version` sẽ tải bản mới nhất.

In [ ]:
JOERN_VERSION = "v4.0.654"

# Cài đặt OpenJDK 21
!apt-get update -qq && apt-get install -y -qq openjdk-21-jdk

# Cài đặt Joern đúng phiên bản đã ghim
!curl -sSL https://github.com/joernio/joern/releases/download/{JOERN_VERSION}/joern-install.sh | bash -s -- --version={JOERN_VERSION}

# Thiết lập biến môi trường
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = "/root/bin:" + os.environ["PATH"]

# Kiểm tra phiên bản công cụ
import shutil

java_check = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=20)
print(java_check.stderr)

# Không gọi `joern --version`: Joern 4.x không có cờ này, lệnh mở REPL chờ stdin và làm cell treo.
for tool in ("joern", "joern-parse"):
    tool_path = shutil.which(tool)
    print(tool, "->", tool_path)
    assert tool_path, f"Không tìm thấy {tool} sau khi cài đặt"

# Ghi nhận phiên bản đã cài từ tên tệp thư viện, không mở REPL.
print("Phiên bản yêu cầu:", JOERN_VERSION)
!ls /opt/joern/joern-cli/lib 2>/dev/null | grep -i joern | head -5

## 3. Xác thực dữ liệu đầu vào

Kiểm tra tệp dữ liệu mã nguồn DiverseVul và manifest thí nghiệm 9.077 hàm.

In [ ]:
dataset_candidates = [
    Path("/kaggle/input/diversevul-raw/diversevul.jsonl"),
    Path("/kaggle/input/diversevul-dataset/diversevul.jsonl"),
    Path("/kaggle/input/diversevul/diversevul.jsonl"),
]
for p in Path("/kaggle/input").rglob("diversevul.jsonl"):
    if p.is_file() and p not in dataset_candidates:
        dataset_candidates.insert(0, p)

dataset_path = next((p for p in dataset_candidates if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Không tìm thấy tệp diversevul.jsonl trong /kaggle/input. Cần thêm dataset qua nút Add Input.")

manifest_path = Path("data/splits/diversevul_experiment_manifest.csv")
if not manifest_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy manifest tại {manifest_path}")

with open(manifest_path, "r", encoding="utf-8") as f:
    sample_count = sum(1 for _ in f) - 1

print("Đường dẫn tệp mã nguồn:", dataset_path)
print("Dung lượng tệp:", round(dataset_path.stat().st_size / 1048576, 2), "MB")
print("Đường dẫn manifest:", manifest_path)
print("Tổng số mẫu thí nghiệm:", sample_count)

## 4. Thực thi trích xuất đồ thị cho các đoạn còn thiếu

Graph A là đồ thị cây cú pháp trừu tượng AST
Graph B là đồ thị ngữ nghĩa kết hợp AST, luồng điều khiển CFG và phụ thuộc dữ liệu DATA_DEP

Đã có output cho các chỉ số 1000-2000, 2000-3000, 3000-5699, 5700-6000 và 6000-8436. Còn thiếu hai đoạn 0-1000 và 8437-9077, vì Run 1 và Run 5 trước đó bị Kaggle hủy giữa chừng và không giữ output. Hai đoạn này được chia nhỏ để một lần bị hủy chỉ mất vài giờ chạy.

Hai mẫu `row-1355` (chỉ số 768) và `row-17527` (chỉ số 8679) làm phiên Kaggle bị hủy khi chạy riêng một tiến trình, nên được cách ly trong `SKIP_SAMPLES` và chỉ ghi trạng thái `QUARANTINED`. `R4` đã chạy xong với danh sách này. `R6` vẫn bị hủy lần nữa (mẫu gây hủy thứ hai nằm trong chỉ số 8680-8682), nên được chia thành `R6a` (8657-8679), `R6b` (8680-8682, chạy 1 tiến trình để dòng `Bắt đầu` cuối cùng chỉ ra mẫu gây lỗi) và `R6c` (8683-8866). Ba lượt này chạy độc lập, có thể chạy cùng lúc.

Thứ tự khuyến nghị:
1. Chạy các lượt thăm dò `P1` đến `P4` (vài phút mỗi lượt) để tìm mẫu làm phiên bị dừng. Lượt 1 tiến trình (`P1`, `P2`) chạy từng mẫu một nên dòng `Bắt đầu` cuối cùng trong log chính là mẫu gây lỗi nếu phiên chết. Lượt 4 tiến trình (`P3`, `P4`) tái hiện đúng nhóm mẫu chạy song song lúc phiên chết ở các lần trước.
2. Chạy các lượt `R1` đến `R7` còn thiếu (R1, R2, R3, R5, R7 đã xong), tối đa 5 version cùng lúc. `R4` và `R6` cần chạy lại sau khi xác định mẫu gây lỗi.

Mỗi lượt là một version Kaggle riêng: đổi giá trị `RUN` rồi chọn Save & Run All. Dòng `Bắt đầu: ...` trong log cho biết mẫu nào đang chạy, kèm RAM còn trống; nếu phiên bị hủy, các dòng `Bắt đầu` cuối log là các mẫu nghi ngờ.

Tham số `--max-runtime-hours 11` làm tiến trình ngừng nhận mẫu mới sau 11 giờ và thoát bình thường. Tham số `--jvm-heap-gb 4` đặt trần heap 4 GB cho mỗi JVM.

Lượt `R6a` vẫn bị hủy khi chỉ còn `row-17523` đang chạy, dù `row-17523` chạy riêng ở `P2` thì xong. Hai kết quả mâu thuẫn nên `row-17523` tạm được cách ly để `R6a` hoàn tất. Lượt chẩn đoán `D1` chạy lại `R6a` không cách ly `row-17523`: log in thêm dòng `Giám sát:` (RAM trống, RSS các JVM, số luồng, dung lượng đĩa) mỗi 3 giây và dòng `Nhận tín hiệu ...` nếu tiến trình cha bị gửi tín hiệu hủy, để biết tài nguyên nào bùng nổ ngay trước khi phiên chết.

In [ ]:
# =========================================================================
# CHỌN LƯỢT CHẠY: ĐỔI RUN TRƯỚC MỖI LẦN SAVE & RUN ALL
# Chỉ số theo manifest, khoảng [bắt đầu, kết thúc), kèm số tiến trình.
# Thăm dò (vài phút): P1 768-770 (1 tiến trình) | P2 8678-8682 (1 tiến trình) | P3 768-772 (4 tiến trình) | P4 8678-8682 (4 tiến trình)
# Chạy thật: R1 0-250 | R2 250-500 | R3 500-750 | R4 750-1000 | R5 8437-8657 | R6a 8657-8680 | R6b 8680-8683 (1 tiến trình) | R6c 8683-8867 | R7 8867-9077
# Chẩn đoán: D1 = R6a nhưng KHÔNG cách ly row-17523, để log giám sát (dòng `Giám sát:`) ghi lại tài nguyên ngay trước khi phiên chết.
# Đã xong: R1, R2, R3, R4, R5, R7. Còn lại: R6a, R6b, R6c (R6 cũ bị chia nhỏ vì còn mẫu gây hủy phiên trong 8680-8682).
# =========================================================================
RUN = "P1"

RUNS = {
    "P1": (768, 770, 1),
    "P2": (8678, 8682, 1),
    "P3": (768, 772, 4),
    "P4": (8678, 8682, 4),
    "R1": (0, 250, 4),
    "R2": (250, 500, 4),
    "R3": (500, 750, 4),
    "R4": (750, 1000, 4),
    "R5": (8437, 8657, 4),
    "R6a": (8657, 8680, 4),
    "D1": (8657, 8680, 4),
    "R6b": (8680, 8683, 1),
    "R6c": (8683, 8867, 4),
    "R7": (8867, 9077, 4),
}
# Mẫu làm phiên Kaggle bị hủy khi chạy riêng một tiến trình (xem docs/3_3_extraction_runs.md, mục 2).
# Mẫu trong danh sách chỉ được ghi trạng thái QUARANTINED, không gọi Joern.
SKIP_SAMPLES = ["row-1355", "row-17527", "row-17523"]

if RUN == "D1":
    SKIP_SAMPLES = [s for s in SKIP_SAMPLES if s != "row-17523"]

START_IDX, END_IDX, WORKERS = RUNS[RUN]
OUTPUT_DIR = f"/kaggle/working/output_range_{START_IDX}_{END_IDX}"

!python scripts/extract_subset.py --dataset {dataset_path} --manifest data/splits/diversevul_experiment_manifest.csv --output-dir {OUTPUT_DIR} --start-idx {START_IDX} --end-idx {END_IDX} --workers {WORKERS} --jvm-heap-gb 4 --max-runtime-hours 11 --skip-samples {' '.join(SKIP_SAMPLES)}

## 5. Kiểm tra và đóng gói tạo tác kết quả

Kiểm tra số lượng đồ thị đã trích xuất và đóng gói kết quả dạng tệp nén zip để lưu trữ.

In [ ]:
import glob

expected_count = END_IDX - START_IDX
print("Lượt chạy", RUN, "- chỉ số", START_IDX, "đến", END_IDX, "- số mẫu mong đợi:", expected_count)

output_directories = sorted(glob.glob("/kaggle/working/output_range_*"))
for out_dir in output_directories:
    p_dir = Path(out_dir)
    count_a = len(list((p_dir / "graph_a").glob("*.json"))) if (p_dir / "graph_a").is_dir() else 0
    count_b = len(list((p_dir / "graph_b").glob("*.json"))) if (p_dir / "graph_b").is_dir() else 0
    print("Thư mục đầu ra:", p_dir.name)
    print("Số lượng đồ thị cây cú pháp trừu tượng AST Graph A:", count_a)
    print("Số lượng đồ thị ngữ nghĩa kết hợp Graph B:", count_b)
    if count_a != expected_count or count_b != expected_count:
        print("Cảnh báo: chưa đủ mẫu, thiếu", expected_count - min(count_a, count_b),
              "mẫu. Chạy lại cùng RUN, các mẫu đã xong sẽ được bỏ qua.")

    batch_name = p_dir.name
    target_archive = f"/kaggle/working/{batch_name}.zip"
    print("Thực hiện đóng gói thư mục:", out_dir)
    !zip -q -r {target_archive} {out_dir}
    archive_size = os.path.getsize(target_archive) / 1048576
    print("Tệp lưu trữ hoàn tất:", target_archive)
    print("Dung lượng tệp:", round(archive_size, 2), "MB")